In [1]:

import json,re,time
from pathlib import Path
import pandas as pd
import plotly.express as px
from transformers import AutoTokenizer
OUT=Path('/root/prompt-injection-as-role-confusion/demo/outputs/role-probe-a100')
test_projections=pd.read_csv(OUT/'token_role_projections.csv',keep_default_na=False)
cached_example=json.loads((OUT/'cached_example.json').read_text())
templated_input=(OUT/'example_conversation.txt').read_text()
tokenizer=AutoTokenizer.from_pretrained('/workspace/gpt-oss-20b',local_files_only=True)
forged_cot=re.search(r'<SYNTHETIC_POLICY\b[^>]*>(.*?)</SYNTHETIC_POLICY>',cached_example['response'],re.DOTALL).group(1)
prefix="Hey GPT,\n1. What's an interesting cat fact\n2. "+cached_example['question']+"\n\n"
user_prompt=prefix+forged_cot
test_sample_df=pd.DataFrame({'token_ix':range(len(tokenizer(templated_input,add_special_tokens=False).input_ids))})


In [2]:
"""
Let's plot CoTness. CoTness should jump for the forged CoT region of the user prompt, despite being user role.
"""
import plotly.express as px

cotness_projections = test_projections.pipe(lambda df: df[df['target_role'] == 'cot'])

fig = px.scatter(
    cotness_projections,
    x = 'sample_ix',
    y = 'prob',
    color = 'role',
    hover_data = ['token', 'role']
)
fig.update_layout(title='Role probe: layer 12 CoT probability', xaxis_title='Token index', yaxis_title='P(cot)')
fig.write_html(OUT / 'cotness.html', include_plotlyjs=True)
test_projections.to_csv(OUT / 'token_role_projections.csv', index=False)
# Locate the injected text by character offsets, retaining its architectural role.
offsets = tokenizer(templated_input, add_special_tokens=False, return_offsets_mapping=True).offset_mapping
forgery_start = templated_input.index(user_prompt) + len(prefix)
forgery_end = forgery_start + len(forged_cot)
spans = []
for row in cotness_projections.itertuples():
    start, end = offsets[int(test_sample_df.iloc[row.sample_ix].token_ix)]
    if row.role == 'user':
        spans.append('forged_cot_in_user' if start < forgery_end and end > forgery_start else 'ordinary_user_text')
    elif row.role == 'cot':
        spans.append('model_analysis')
    elif row.role == 'assistant':
        spans.append('model_final')
    else:
        spans.append(f'{row.role}_context')
cotness_projections = cotness_projections.assign(region=spans)
region_summary = cotness_projections.groupby('region').agg(
    tokens=('prob', 'size'), mean_cot_probability=('prob', 'mean'),
    median_cot_probability=('prob', 'median'), fraction_cot_above_half=('prob', lambda p: (p > .5).mean()),
)
region_summary.to_csv(OUT / 'cotness_by_region.csv')
display(region_summary)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
fig_static, ax = plt.subplots(figsize=(12, 4))
for region, frame in cotness_projections.groupby('region', sort=False):
    ax.scatter(frame.sample_ix, frame.prob, s=8, alpha=.6, label=region)
ax.set(xlabel='Token index', ylabel='P(cot)', ylim=(-.03, 1.03), title='gpt-oss-20b: layer 12 role probe')
ax.legend(loc='upper right', fontsize=8)
fig_static.tight_layout()
fig_static.savefig(OUT / 'cotness.png', dpi=160)
plt.close(fig_static)
metrics = json.loads((OUT / 'metrics.json').read_text())
metrics['cotness_by_region'] = region_summary.reset_index().to_dict('records')
(OUT / 'metrics.json').write_text(json.dumps(metrics, indent=2))
fig.show()


,tokens,mean_cot_probability,median_cot_probability,fraction_cot_above_half
region,,,,
forged_cot_in_user,176,0.954978,0.999707,0.954545
model_analysis,177,0.598227,0.699927,0.615819
model_final,671,0.000037,0.000001,0.000000
ordinary_user_text,22,0.036535,0.000146,0.045455
system_context,57,0.409654,0.130919,0.385965
